# Function 3: Part 2 analysis notebook

This notebook keeps the original data-loading cells, appends the latest query point/output from the end of your uploaded notebook, and then runs one focused analysis for Part 2.

Assumption: lower output is better, so the optimisation target is minimisation.


In [ ]:
import numpy as np

input_data = np.load('../../data/initial_data/function_3/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([
    [0.754364, 0.233177, 0.303208],
    [0.312229, 0.060777, 0.000904],
    [0.5, 0.5, 0.5],
    ])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


Before: (15, 3)
After: (17, 3)
[[1.71525207e-01 3.43916870e-01 2.48737201e-01]
 [2.42114461e-01 6.44074270e-01 2.72432809e-01]
 [5.34905720e-01 3.98500915e-01 1.73388729e-01]
 [4.92581415e-01 6.11593188e-01 3.40176386e-01]
 [1.34621666e-01 2.19917240e-01 4.58206220e-01]
 [3.45523271e-01 9.41359831e-01 2.69363479e-01]
 [1.51836632e-01 4.39990619e-01 9.90881867e-01]
 [6.45502835e-01 3.97142940e-01 9.19771338e-01]
 [7.46911945e-01 2.84196309e-01 2.26299855e-01]
 [1.70476994e-01 6.97032401e-01 1.49169434e-01]
 [2.20549337e-01 2.97825244e-01 3.43555344e-01]
 [6.66013659e-01 6.71985151e-01 2.46295297e-01]
 [4.68089497e-02 2.31360241e-01 7.70617592e-01]
 [6.00097282e-01 7.25135725e-01 6.60886415e-02]
 [9.65994849e-01 8.61119690e-01 5.66829131e-01]
 [7.54364000e-01 2.33177000e-01 3.03208000e-01]
 [3.12229000e-01 6.07770000e-02 9.04000000e-04]]


In [ ]:
output_data = np.load('../../data/initial_data/function_3/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([
    -0.09200841551496666,
    -0.18083748652026374,
    -0.015979341188442648
    ])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


Before: (15,)
After: (17,)
[-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837 -0.09200842 -0.18083749]


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Latest query/output extracted from the uploaded notebook.
# Edit these if you later receive a different portal output.
latest_query = np.array([[0.5, 0.5, 0.5]])
actual_output = -0.015979341188442648

# Append the latest query/output only if it is not already present.
if actual_output is not None:
    already_present = np.any(np.all(np.isclose(input_data, latest_query, atol=1e-12), axis=1))
    if not already_present:
        input_data = np.vstack([input_data, latest_query])
        output_data = np.append(output_data, actual_output)

function_id = 3
d = input_data.shape[1]
print(f"Function {function_id}, dimension d={d}")
print("Data shape:", input_data.shape, output_data.shape)
print("Current best observed y:", output_data.min())
print("Current best x:", input_data[np.argmin(output_data)])


Function 3, dimension d=3
Data shape: (18, 3) (18,)
Current best observed y: -0.3989255131463011
Current best x: [0.15183663 0.43999062 0.99088187]


In [4]:
# Basic table used in all interpretations
summary = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
summary["y"] = output_data
summary["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
summary["rank_min"] = summary["y"].rank(method="first", ascending=True).astype(int)
summary = summary.sort_values("y")
display(summary)


,x1,x2,x3,y,log_abs_y,rank_min
6,0.151837,0.439991,0.990882,-0.398926,-0.918981,1
16,0.312229,0.060777,0.000904,-0.180837,-1.710157,2
8,0.746912,0.284196,0.226300,-0.131461,-2.029048,3
12,0.046809,0.231360,0.770618,-0.118048,-2.136662,4
7,0.645503,0.397143,0.919771,-0.113869,-2.172711,5
0,0.171525,0.343917,0.248737,-0.112122,-2.188166,6
2,0.534906,0.398501,0.173389,-0.111415,-2.194496,7
5,0.345523,0.941360,0.269363,-0.110621,-2.201646,8
11,0.666014,0.671985,0.246295,-0.105965,-2.244646,9
9,0.170477,0.697032,0.149169,-0.094190,-2.362446,10


## Gaussian process surrogate + expected improvement for minimisation

In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm

# Standardise inputs and outputs for numerical stability.
X = input_data.copy()
y = output_data.copy()

x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e2), nu=2.5) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e-1))

gp = GaussianProcessRegressor(kernel=kernel, normalize_y=False, n_restarts_optimizer=20, random_state=0)
gp.fit(X_scaled, y_scaled)

print("Fitted kernel:", gp.kernel_)
print("Best observed y:", y.min())
print("Best observed x:", X[np.argmin(y)])


Fitted kernel: 1.49**2 * Matern(length_scale=[100, 5.72, 0.368], nu=2.5) + WhiteKernel(noise_level=4.29e-10)
Best observed y: -0.3989255131463011
Best observed x: [0.15183663 0.43999062 0.99088187]


/Users/andriy/miniconda3/envs/appenv/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [6]:
def expected_improvement_min(X_candidates, gp, y_best_scaled, xi=0.01):
    """Expected improvement for minimisation in scaled y-space."""
    mu, sigma = gp.predict(X_candidates, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    improvement = y_best_scaled - mu - xi
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei

# Random candidate search in [0, 1]^d. Increase n_candidates for a more exhaustive search.
rng = np.random.default_rng(0)
n_candidates = 20000 if d <= 4 else 50000
candidates = rng.random((n_candidates, d))
candidates_scaled = x_scaler.transform(candidates)

y_best_scaled = y_scaled.min()
ei = expected_improvement_min(candidates_scaled, gp, y_best_scaled, xi=0.01)
mu_scaled, std_scaled = gp.predict(candidates_scaled, return_std=True)
mu = y_scaler.inverse_transform(mu_scaled.reshape(-1, 1)).ravel()
std = std_scaled * y_scaler.scale_[0]

results = pd.DataFrame(candidates, columns=[f"x{i+1}" for i in range(d)])
results["pred_mean"] = mu
results["pred_std"] = std
results["expected_improvement"] = ei
results = results.sort_values("expected_improvement", ascending=False)

display(results.head(10))

best_next = results.iloc[0][[f"x{i+1}" for i in range(d)]].to_numpy(dtype=float)
print("Suggested next query:", np.round(best_next, 6))
print("Portal format:", ", ".join(f"x{i+1}={v:.6f}" for i, v in enumerate(best_next)))


,x1,x2,x3,pred_mean,pred_std,expected_improvement
3326,0.008443,0.441970,0.999774,-0.417036,0.011002,0.214880
18270,0.314315,0.381055,0.999878,-0.416134,0.013197,0.208806
176,0.717241,0.484515,0.999501,-0.416244,0.012252,0.208024
3707,0.742565,0.464985,0.999438,-0.416217,0.011508,0.206337
15975,0.597708,0.675780,0.999701,-0.409206,0.028758,0.205751
10330,0.315875,0.605834,0.999384,-0.412878,0.021517,0.204322
18899,0.415349,0.747138,0.999868,-0.404045,0.035882,0.202677
1195,0.206065,0.525372,0.999104,-0.415247,0.013980,0.201184
16995,0.838289,0.474717,0.999202,-0.415716,0.011754,0.201161
13299,0.954128,0.675072,0.999412,-0.408518,0.028838,0.200870


Suggested next query: [0.008443 0.44197  0.999774]
Portal format: x1=0.008443, x2=0.441970, x3=0.999774


In [8]:
# Optional 2D visualisation only when d=2
if d == 2:
    grid_res = 150
    xx, yy = np.meshgrid(np.linspace(0, 1, grid_res), np.linspace(0, 1, grid_res))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = x_scaler.transform(grid)
    mu_grid_scaled, std_grid_scaled = gp.predict(grid_scaled, return_std=True)
    mu_grid = y_scaler.inverse_transform(mu_grid_scaled.reshape(-1, 1)).reshape(grid_res, grid_res)
    ei_grid = expected_improvement_min(grid_scaled, gp, y_best_scaled).reshape(grid_res, grid_res)

    plt.figure(figsize=(7, 6))
    cf = plt.contourf(xx, yy, mu_grid, levels=40)
    plt.colorbar(cf, label="GP predicted mean")
    plt.scatter(input_data[:, 0], input_data[:, 1], c=output_data, edgecolors="black", s=80)
    plt.scatter(best_next[0], best_next[1], marker="*", s=250, edgecolors="black", label="suggested next")
    plt.xlabel("x1"); plt.ylabel("x2"); plt.title("GP predicted mean")
    plt.legend(); plt.show()

    plt.figure(figsize=(7, 6))
    cf = plt.contourf(xx, yy, ei_grid, levels=40)
    plt.colorbar(cf, label="Expected improvement")
    plt.scatter(input_data[:, 0], input_data[:, 1], c="white", edgecolors="black", s=80)
    plt.scatter(best_next[0], best_next[1], marker="*", s=250, edgecolors="black", label="suggested next")
    plt.xlabel("x1"); plt.ylabel("x2"); plt.title("Expected improvement for minimisation")
    plt.legend(); plt.show()
